# Homework: Contrast Stretching

For each pixel value `x`:

| Condition | New value |
|---|---|
| `x ≤ low` | `0` |
| `x ≥ high` | `255` |
| `low < x < high` (middle range) | `255 * (x - low) / (high - low)` |

## 1. Upload an image and convert it to black & white (grayscale)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

uploaded = files.upload()            # choose any image from your computer
filename = next(iter(uploaded))

pil_img = Image.open(filename).convert('L')   # 'L' = 8-bit grayscale (0..255)
pil_img.thumbnail((800, 800))                 # shrink big photos so the pixel loop runs fast
img = np.array(pil_img)

print('Image size:', img.shape)
print('Min pixel:', img.min(), '  Max pixel:', img.max())

## 2. Look at the histogram to choose `low` and `high`

In [ ]:
plt.figure(figsize=(8, 3))
plt.hist(img.ravel(), bins=256, range=(0, 255), color='gray')
plt.title('Histogram of original image')
plt.xlabel('Pixel value')
plt.ylabel('Count')
plt.show()

In [ ]:
# Pick the range where most pixels are (see histogram above)
low = 60
high = 180

# Or choose them automatically (ignores the darkest/brightest 2% of pixels):
# low, high = np.percentile(img, (2, 98)).astype(int)

print('low =', low, ' high =', high)

## 3. Apply contrast stretching to each pixel

In [ ]:
def contrast_stretch(img, low, high):
    out = np.zeros_like(img)
    rows, cols = img.shape

    for i in range(rows):
        for j in range(cols):
            x = int(img[i, j])

            if x <= low:
                out[i, j] = 0
            elif x >= high:
                out[i, j] = 255
            else:  # middle-range pixels: low < x < high
                out[i, j] = round(255 * (x - low) / (high - low))

    return out

stretched = contrast_stretch(img, low, high)
print('New min:', stretched.min(), '  New max:', stretched.max())

## 4. Compare before and after

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 9))

# vmin/vmax fixed to 0..255 so matplotlib does not auto-stretch the original
ax[0, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
ax[0, 0].set_title('Original')
ax[0, 0].axis('off')

ax[0, 1].imshow(stretched, cmap='gray', vmin=0, vmax=255)
ax[0, 1].set_title(f'Contrast stretched (low={low}, high={high})')
ax[0, 1].axis('off')

ax[1, 0].hist(img.ravel(), bins=256, range=(0, 255), color='gray')
ax[1, 0].set_title('Histogram - original')

ax[1, 1].hist(stretched.ravel(), bins=256, range=(0, 255), color='gray')
ax[1, 1].set_title('Histogram - stretched')

plt.tight_layout()
plt.show()

## 5. (Optional) Download the result

In [ ]:
Image.fromarray(stretched).save('stretched.png')
files.download('stretched.png')